# Example 10.1.1 / 1: A direct network declaration

Companion notebook for **10.1.1 node and arc Declarations** in the AMPL lab Documentation, section "A direct network declaration". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A direct network declaration

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NODES;  # supply, transfer and demand points
set ARCS within NODES cross NODES;  # permitted directed connections
param balance {NODES};  # net supply; negative means net demand
param capacity {ARCS} >= 0;  # maximum flow on the arc
param unit_cost {ARCS} >= 0;  # cost per unit of flow on the arc

minimize TotalCost;  # terms are added by the arc declaration

node Balance {k in NODES}:  # net outflow equals net supply
    net_out = balance[k];

# flow i to j: out of Balance[i], into Balance[j]
arc Flow {(i, j) in ARCS} >= 0, <= capacity[i, j],
    from Balance[i], to Balance[j],
    obj TotalCost unit_cost[i, j];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["NODES"] = ["source", "hub", "sink"]
ampl.set["ARCS"] = [("source", "hub"), ("hub", "sink"), ("source", "sink")]
ampl.param["balance"] = {"source": 5, "hub": 0, "sink": -5}
ampl.param["capacity"] = {("source", "hub"): 5, ("hub", "sink"): 5, ("source", "sink"): 5}
ampl.param["unit_cost"] = {("source", "hub"): 1, ("hub", "sink"): 1, ("source", "sink"): 4}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.get_data("Flow").to_dict())

HiGHS 1.14.0: optimal solution; objective 10
0 simplex iterations
0 barrier iterations
solved
{('hub', 'sink'): 5, ('source', 'hub'): 5, ('source', 'sink'): 0}


### Pure AMPL: data.dat and model.run

The same declarations can use pure-AMPL data and commands:

In [6]:
%%writefile data.dat
set NODES := source hub sink;
set ARCS := (source,hub) (hub,sink) (source,sink);
param balance := source 5 hub 0 sink -5;
param capacity := [source,hub] 5 [hub,sink] 5 [source,sink] 5;
param unit_cost := [source,hub] 1 [hub,sink] 1 [source,sink] 4;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Flow;

HiGHS 1.14.0: optimal solution; objective 10
0 simplex iterations
0 barrier iterations
Flow :=
hub    sink   5
source hub    5
source sink   0
;

